# 🎨 ULTRON Agent - Stable Diffusion Integration
## Advanced Image Generation with API Interface

This notebook provides a complete Stable Diffusion implementation that integrates with the ULTRON Agent system.

### Features:
- 🎨 High-quality image generation using Stable Diffusion
- 🌐 REST API endpoints for ULTRON Agent integration
- 🖼️ Image gallery and management
- ⚙️ Advanced parameter controls
- 📱 Interactive GUI interface
- 🔄 Real-time generation monitoring


In [ ]:
# 🔧 Environment Setup and Dependencies Installation
import subprocess
import sys
import os
from pathlib import Path

def install_package(package):
    """Install package with progress indication"""
    print(f"📦 Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
    print(f"✅ {package} installed successfully")

# Essential packages for Stable Diffusion
packages = [
    "diffusers>=0.25.0",
    "transformers>=4.36.0", 
    "torch>=2.1.0",
    "torchvision>=0.16.0",
    "accelerate>=0.25.0",
    "xformers",
    "controlnet-aux",
    "opencv-python",
    "pillow>=10.0.0",
    "numpy>=1.24.0",
    "flask>=3.0.0",
    "flask-cors>=4.0.0",
    "requests>=2.31.0",
    "gradio>=4.0.0",
    "ipywidgets>=8.0.0",
    "matplotlib>=3.7.0",
    "pyngrok"
]

print("🚀 Starting ULTRON Stable Diffusion Environment Setup...")
for package in packages:
    try:
        install_package(package)
    except Exception as e:
        print(f"⚠️ Warning: Failed to install {package}: {e}")

print("\n🎉 Environment setup complete!")

In [ ]:
# 📚 Import Required Libraries
import torch
import torchvision.transforms as transforms
from diffusers import (
    StableDiffusionPipeline, 
    StableDiffusionImg2ImgPipeline,
    StableDiffusionInpaintPipeline,
    DPMSolverMultistepScheduler,
    EulerAncestralDiscreteScheduler,
    DDIMScheduler
)
from PIL import Image, ImageEnhance, ImageFilter
import numpy as np
import matplotlib.pyplot as plt
import base64
import io
import json
import time
import random
import threading
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import gradio as gr
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
import uuid
import os
from pathlib import Path

# Configure device and memory optimization
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Using device: {device}")
if device == "cuda":
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Memory optimization settings
torch.backends.cudnn.benchmark = True
if device == "cuda":
    torch.cuda.empty_cache()

print("✅ Libraries imported successfully!")

In [ ]:
# 🎨 Advanced Stable Diffusion Manager
class UltronStableDiffusion:
    """Advanced Stable Diffusion interface for ULTRON Agent"""
    
    def __init__(self, model_id="runwayml/stable-diffusion-v1-5", device="auto"):
        self.device = device if device != "auto" else ("cuda" if torch.cuda.is_available() else "cpu")
        self.model_id = model_id
        self.pipe = None
        self.img2img_pipe = None
        self.inpaint_pipe = None
        
        # Image storage and management
        self.images_dir = Path("ultron_generated_images")
        self.images_dir.mkdir(exist_ok=True)
        self.generation_history = []
        self.current_session = str(uuid.uuid4())[:8]
        
        # Available models
        self.available_models = {
            "Stable Diffusion v1.5": "runwayml/stable-diffusion-v1-5",
            "Stable Diffusion v2.1": "stabilityai/stable-diffusion-2-1",
            "Dreamlike Photoreal": "dreamlike-art/dreamlike-photoreal-2.0",
            "Realistic Vision": "SG161222/Realistic_Vision_V4.0",
            "Anything v4": "andite/anything-v4.0"
        }
        
        print(f"🚀 ULTRON Stable Diffusion initialized on {self.device}")
        print(f"📁 Images will be saved to: {self.images_dir.absolute()}")
    
    def load_model(self, model_id=None, scheduler="DPMSolver"):
        """Load Stable Diffusion model with optimizations"""
        if model_id:
            self.model_id = model_id
            
        print(f"📦 Loading model: {self.model_id}...")
        
        try:
            # Load with memory optimizations
            self.pipe = StableDiffusionPipeline.from_pretrained(
                self.model_id,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
                safety_checker=None,
                requires_safety_checker=False
            )
            
            # Apply scheduler
            if scheduler == "DPMSolver":
                self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(self.pipe.scheduler.config)
            elif scheduler == "Euler":
                self.pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(self.pipe.scheduler.config)
            elif scheduler == "DDIM":
                self.pipe.scheduler = DDIMScheduler.from_config(self.pipe.scheduler.config)
            
            # Move to device and optimize
            self.pipe = self.pipe.to(self.device)
            
            if self.device == "cuda":
                self.pipe.enable_model_cpu_offload()
                self.pipe.enable_xformers_memory_efficient_attention()
                
            print(f"✅ Model loaded successfully with {scheduler} scheduler!")
            return True
            
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            return False
    
    def generate_image(self, 
                      prompt: str,
                      negative_prompt: str = "ugly, blurry, poor quality, distorted",
                      width: int = 512,
                      height: int = 512,
                      num_inference_steps: int = 20,
                      guidance_scale: float = 7.5,
                      num_images: int = 1,
                      seed: Optional[int] = None) -> List[Dict]:
        """Generate images using Stable Diffusion"""
        
        if not self.pipe:
            if not self.load_model():
                return [{"error": "Failed to load model"}]
        
        if seed is None:
            seed = random.randint(0, 2**32)
        
        generator = torch.Generator(device=self.device).manual_seed(seed)
        
        print(f"🎨 Generating {num_images} image(s) with prompt: '{prompt[:50]}...'")
        start_time = time.time()
        
        try:
            # Generate images
            with torch.autocast(self.device):
                result = self.pipe(
                    prompt=prompt,
                    negative_prompt=negative_prompt,
                    width=width,
                    height=height,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale,
                    num_images_per_prompt=num_images,
                    generator=generator
                )
            
            generation_time = time.time() - start_time
            generated_images = []
            
            # Save and process images
            for i, image in enumerate(result.images):
                # Create unique filename
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"ultron_sd_{timestamp}_{self.current_session}_{i:02d}.png"
                filepath = self.images_dir / filename
                
                # Save image
                image.save(filepath)
                
                # Convert to base64 for API
                buffered = io.BytesIO()
                image.save(buffered, format="PNG")
                img_base64 = base64.b64encode(buffered.getvalue()).decode()
                
                image_data = {
                    "filename": filename,
                    "filepath": str(filepath),
                    "base64": img_base64,
                    "prompt": prompt,
                    "negative_prompt": negative_prompt,
                    "seed": seed,
                    "width": width,
                    "height": height,
                    "steps": num_inference_steps,
                    "guidance_scale": guidance_scale,
                    "generation_time": generation_time,
                    "timestamp": timestamp,
                    "session": self.current_session
                }
                
                generated_images.append(image_data)
                self.generation_history.append(image_data)
            
            print(f"✅ Generated {len(generated_images)} image(s) in {generation_time:.2f}s")
            return generated_images
            
        except Exception as e:
            print(f"❌ Generation error: {e}")
            return [{"error": str(e)}]
    
    def get_generation_history(self, limit: int = 50) -> List[Dict]:
        """Get recent generation history"""
        return self.generation_history[-limit:]
    
    def clear_history(self):
        """Clear generation history"""
        self.generation_history.clear()
        print("🗑️ Generation history cleared")
    
    def get_available_models(self) -> Dict[str, str]:
        """Get list of available models"""
        return self.available_models

# Initialize the Stable Diffusion system
ultron_sd = UltronStableDiffusion()
print("🎉 ULTRON Stable Diffusion system ready!")

In [ ]:
# 🌐 Flask API Server for ULTRON Agent Integration
app = Flask(__name__)
CORS(app)

@app.route('/health', methods=['GET'])
def health_check():
    """Health check endpoint"""
    return jsonify({
        "status": "healthy",
        "service": "ULTRON Stable Diffusion",
        "device": ultron_sd.device,
        "model": ultron_sd.model_id,
        "timestamp": datetime.now().isoformat()
    })

@app.route('/generate', methods=['POST'])
def generate_image_api():
    """Generate image via API"""
    try:
        data = request.get_json()
        
        # Extract parameters
        prompt = data.get('prompt', '')
        if not prompt:
            return jsonify({"error": "Prompt is required"}), 400
        
        negative_prompt = data.get('negative_prompt', 'ugly, blurry, poor quality')
        width = data.get('width', 512)
        height = data.get('height', 512)
        steps = data.get('steps', 20)
        guidance_scale = data.get('guidance_scale', 7.5)
        num_images = data.get('num_images', 1)
        seed = data.get('seed')
        
        # Generate images
        results = ultron_sd.generate_image(
            prompt=prompt,
            negative_prompt=negative_prompt,
            width=width,
            height=height,
            num_inference_steps=steps,
            guidance_scale=guidance_scale,
            num_images=num_images,
            seed=seed
        )
        
        if results and "error" in results[0]:
            return jsonify({"error": results[0]["error"]}), 500
        
        return jsonify({
            "success": True,
            "images": results,
            "count": len(results)
        })
        
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/history', methods=['GET'])
def get_history_api():
    """Get generation history"""
    limit = request.args.get('limit', 50, type=int)
    history = ultron_sd.get_generation_history(limit)
    return jsonify({
        "history": history,
        "count": len(history)
    })

@app.route('/models', methods=['GET'])
def get_models_api():
    """Get available models"""
    models = ultron_sd.get_available_models()
    return jsonify({"models": models})

@app.route('/switch_model', methods=['POST'])
def switch_model_api():
    """Switch Stable Diffusion model"""
    try:
        data = request.get_json()
        model_id = data.get('model_id')
        scheduler = data.get('scheduler', 'DPMSolver')
        
        if not model_id:
            return jsonify({"error": "model_id is required"}), 400
        
        success = ultron_sd.load_model(model_id, scheduler)
        
        if success:
            return jsonify({
                "success": True,
                "message": f"Switched to model: {model_id}",
                "current_model": ultron_sd.model_id
            })
        else:
            return jsonify({"error": "Failed to load model"}), 500
            
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/image/<filename>', methods=['GET'])
def get_image(filename):
    """Serve generated images"""
    try:
        filepath = ultron_sd.images_dir / filename
        if filepath.exists():
            return send_file(filepath)
        else:
            return jsonify({"error": "Image not found"}), 404
    except Exception as e:
        return jsonify({"error": str(e)}), 500

print("🌐 API Server configured successfully!")

In [ ]:
# 🎮 Interactive Gradio Interface
def create_gradio_interface():
    """Create interactive Gradio interface for Stable Diffusion"""
    
    def generate_gradio(prompt, negative_prompt, width, height, steps, guidance_scale, num_images, seed_input, model_choice):
        """Generate images through Gradio interface"""
        try:
            # Switch model if needed
            if model_choice != "Current Model" and model_choice in ultron_sd.available_models:
                model_id = ultron_sd.available_models[model_choice]
                ultron_sd.load_model(model_id)
            
            # Parse seed
            seed = None if seed_input == -1 else seed_input
            
            # Generate images
            results = ultron_sd.generate_image(
                prompt=prompt,
                negative_prompt=negative_prompt,
                width=width,
                height=height,
                num_inference_steps=steps,
                guidance_scale=guidance_scale,
                num_images=num_images,
                seed=seed
            )
            
            if results and "error" not in results[0]:
                # Convert base64 to PIL Images for Gradio
                images = []
                for result in results:
                    img_data = base64.b64decode(result['base64'])
                    img = Image.open(io.BytesIO(img_data))
                    images.append(img)
                
                return images, f"✅ Generated {len(images)} image(s) successfully!"
            else:
                error_msg = results[0].get('error', 'Unknown error') if results else 'No results'
                return [], f"❌ Error: {error_msg}"
                
        except Exception as e:
            return [], f"❌ Exception: {str(e)}"
    
    def load_history():
        """Load recent generation history"""
        history = ultron_sd.get_generation_history(10)
        if history:
            images = []
            captions = []
            for item in reversed(history):
                try:
                    img_data = base64.b64decode(item['base64'])
                    img = Image.open(io.BytesIO(img_data))
                    images.append(img)
                    captions.append(f"{item['prompt'][:50]}... | {item['timestamp']}")
                except:
                    continue
            return list(zip(images, captions)) if images else []
        return []
    
    # Create interface
    with gr.Blocks(title="🎨 ULTRON Stable Diffusion", theme=gr.themes.Soft()) as demo:
        gr.Markdown("""
        # 🎨 ULTRON Agent - Stable Diffusion Interface
        ### Advanced AI Image Generation System
        Generate high-quality images using state-of-the-art Stable Diffusion models.
        """)
        
        with gr.Tab("🎨 Generate Images"):
            with gr.Row():
                with gr.Column(scale=2):
                    prompt = gr.Textbox(
                        label="Prompt",
                        placeholder="Describe the image you want to generate...",
                        lines=3
                    )
                    negative_prompt = gr.Textbox(
                        label="Negative Prompt",
                        value="ugly, blurry, poor quality, distorted, deformed",
                        lines=2
                    )
                    
                    with gr.Row():
                        width = gr.Slider(256, 1024, value=512, step=64, label="Width")
                        height = gr.Slider(256, 1024, value=512, step=64, label="Height")
                    
                    with gr.Row():
                        steps = gr.Slider(1, 50, value=20, step=1, label="Inference Steps")
                        guidance_scale = gr.Slider(1.0, 20.0, value=7.5, step=0.5, label="Guidance Scale")
                    
                    with gr.Row():
                        num_images = gr.Slider(1, 4, value=1, step=1, label="Number of Images")
                        seed_input = gr.Number(label="Seed (-1 for random)", value=-1)
                    
                    model_choice = gr.Dropdown(
                        choices=["Current Model"] + list(ultron_sd.available_models.keys()),
                        value="Current Model",
                        label="Model Selection"
                    )
                    
                    generate_btn = gr.Button("🎨 Generate Images", variant="primary", size="lg")
                    status = gr.Textbox(label="Status", interactive=False)
                
                with gr.Column(scale=3):
                    output_gallery = gr.Gallery(
                        label="Generated Images",
                        show_label=True,
                        elem_id="gallery",
                        columns=2,
                        rows=2,
                        height="auto"
                    )
        
        with gr.Tab("📖 History"):
            with gr.Row():
                refresh_btn = gr.Button("🔄 Refresh History", variant="secondary")
                clear_btn = gr.Button("🗑️ Clear History", variant="stop")
            
            history_gallery = gr.Gallery(
                label="Generation History",
                show_label=True,
                columns=3,
                rows=3,
                height="auto"
            )
        
        with gr.Tab("⚙️ System Info"):
            system_info = gr.Markdown(f"""
            ### 🤖 ULTRON Stable Diffusion System Status
            
            **Device:** {ultron_sd.device}  
            **Current Model:** {ultron_sd.model_id}  
            **Images Directory:** {ultron_sd.images_dir}  
            **Session ID:** {ultron_sd.current_session}  
            **Generation Count:** {len(ultron_sd.generation_history)}  
            
            ### 🎯 Available Models:
            {chr(10).join([f'- **{name}:** `{model_id}`' for name, model_id in ultron_sd.available_models.items()])}
            """)
        
        # Event handlers
        generate_btn.click(
            fn=generate_gradio,
            inputs=[prompt, negative_prompt, width, height, steps, guidance_scale, num_images, seed_input, model_choice],
            outputs=[output_gallery, status]
        )
        
        refresh_btn.click(
            fn=load_history,
            outputs=[history_gallery]
        )
        
        clear_btn.click(
            fn=lambda: (ultron_sd.clear_history(), []),
            outputs=[history_gallery]
        )
    
    return demo

print("🎮 Gradio interface configured!")

In [ ]:
# 🚀 Launch Servers and Interfaces
import threading
from IPython.display import display, HTML
import time

# Configuration
API_PORT = 8000
GRADIO_PORT = 7860

def start_api_server():
    """Start Flask API server in background"""
    try:
        print(f"🌐 Starting API server on port {API_PORT}...")
        app.run(host='0.0.0.0', port=API_PORT, debug=False, use_reloader=False)
    except Exception as e:
        print(f"❌ API server error: {e}")

def start_gradio_interface():
    """Start Gradio interface"""
    try:
        print(f"🎮 Starting Gradio interface on port {GRADIO_PORT}...")
        demo = create_gradio_interface()
        demo.launch(
            server_name="0.0.0.0",
            server_port=GRADIO_PORT,
            share=True,  # Create public link for Colab
            debug=False,
            show_error=True
        )
    except Exception as e:
        print(f"❌ Gradio interface error: {e}")

# Load initial model
print("📦 Loading initial Stable Diffusion model...")
if ultron_sd.load_model():
    print("✅ Model loaded successfully!")
    
    # Start API server in background thread
    api_thread = threading.Thread(target=start_api_server, daemon=True)
    api_thread.start()
    
    # Wait a moment for API server to start
    time.sleep(3)
    
    # Display connection information
    display(HTML(f"""
    <div style="padding: 20px; background: linear-gradient(45deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 10px; margin: 10px 0;">
        <h2>🎉 ULTRON Stable Diffusion System Ready!</h2>
        <p><strong>🌐 API Endpoint:</strong> <code>http://localhost:{API_PORT}</code></p>
        <p><strong>🎮 Gradio Interface:</strong> Starting below...</p>
        <p><strong>📖 API Documentation:</strong></p>
        <ul>
            <li><code>GET /health</code> - Health check</li>
            <li><code>POST /generate</code> - Generate images</li>
            <li><code>GET /history</code> - Get generation history</li>
            <li><code>GET /models</code> - List available models</li>
            <li><code>POST /switch_model</code> - Switch models</li>
        </ul>
    </div>
    """))
    
    # Start Gradio interface (this will block)
    start_gradio_interface()
    
else:
    print("❌ Failed to load initial model. Please check your setup.")

In [ ]:
# 🧪 Test ULTRON Agent Integration
def test_ultron_integration():
    """Test the integration with ULTRON Agent"""
    import requests
    import json
    
    base_url = f"http://localhost:{API_PORT}"
    
    print("🧪 Testing ULTRON Stable Diffusion API Integration...")
    
    # Test health check
    try:
        response = requests.get(f"{base_url}/health")
        if response.status_code == 200:
            print("✅ Health check passed")
            print(f"   Response: {response.json()}")
        else:
            print(f"❌ Health check failed: {response.status_code}")
    except Exception as e:
        print(f"❌ Health check error: {e}")
    
    # Test image generation
    try:
        test_prompt = "A futuristic AI robot assistant, cyberpunk style, high quality, detailed"
        payload = {
            "prompt": test_prompt,
            "negative_prompt": "ugly, blurry, poor quality",
            "width": 512,
            "height": 512,
            "steps": 10,  # Quick test
            "guidance_scale": 7.5,
            "num_images": 1
        }
        
        print(f"🎨 Testing image generation with prompt: '{test_prompt}'")
        response = requests.post(f"{base_url}/generate", json=payload, timeout=120)
        
        if response.status_code == 200:
            result = response.json()
            if result.get('success'):
                print("✅ Image generation test passed")
                print(f"   Generated {result.get('count')} image(s)")
                
                # Display first image if available
                if result.get('images'):
                    img_data = result['images'][0]
                    print(f"   Image saved as: {img_data['filename']}")
                    
                    # Decode and display
                    img_bytes = base64.b64decode(img_data['base64'])
                    img = Image.open(io.BytesIO(img_bytes))
                    
                    plt.figure(figsize=(8, 8))
                    plt.imshow(img)
                    plt.axis('off')
                    plt.title(f"Generated Image\nPrompt: {test_prompt[:50]}...")
                    plt.show()
            else:
                print(f"❌ Image generation failed: {result}")
        else:
            print(f"❌ Image generation request failed: {response.status_code}")
            print(f"   Response: {response.text}")
            
    except Exception as e:
        print(f"❌ Image generation test error: {e}")
    
    # Test history endpoint
    try:
        response = requests.get(f"{base_url}/history")
        if response.status_code == 200:
            history = response.json()
            print(f"✅ History test passed - {history.get('count')} items in history")
        else:
            print(f"❌ History test failed: {response.status_code}")
    except Exception as e:
        print(f"❌ History test error: {e}")
    
    print("\n🎉 Integration testing complete!")

# Note: Uncomment the line below to run the test
# test_ultron_integration()

# 📖 Usage Instructions

## 🚀 Getting Started

1. **Run all cells above** to set up the Stable Diffusion system
2. **Wait for model loading** - This may take a few minutes on first run
3. **Access the interfaces:**
   - **Gradio UI**: Use the interface that appears below
   - **API Endpoints**: Available at `http://localhost:8000`

## 🎨 Using the System

### Through Gradio Interface:
- Enter your prompt in the text box
- Adjust parameters as needed
- Click "Generate Images"
- View results in the gallery

### Through API (for ULTRON Agent):
```python
import requests

# Generate image
response = requests.post('http://localhost:8000/generate', json={
    "prompt": "A beautiful landscape with mountains",
    "negative_prompt": "ugly, blurry",
    "width": 512,
    "height": 512,
    "steps": 20,
    "guidance_scale": 7.5
})

result = response.json()
```

## 🔧 Integration with ULTRON Agent

This notebook creates API endpoints that your ULTRON Agent can connect to:

- **Health Check**: `GET /health`
- **Generate Images**: `POST /generate`
- **View History**: `GET /history`
- **Switch Models**: `POST /switch_model`
- **Get Image**: `GET /image/<filename>`

## 💡 Tips

- Use detailed prompts for better results
- Adjust guidance scale (7.5 is usually good)
- Higher steps = better quality but slower generation
- Try different models for different styles
- Use negative prompts to avoid unwanted elements
